In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [ ]:
df_encounters=spark.read.csv(
    
    "Files/encounters.csv", header=True)

display(df_encounters)

In [ ]:

from pyspark.sql.functions import col, lit, when, min, datediff, to_date
from pyspark.sql.window import Window

df_encounters_transformed = (
    df_encounters
    .withColumn(
        "AdmittedDays",
        datediff(to_date("STOP"), to_date("START"))
    )
    .withColumn(
        "MinDate",
        min("START").over(
            Window.partitionBy("PATIENT").orderBy("START")
        )
    )
    .withColumn(
        "FirstVisit",
        when(col("MinDate") == col("START"), lit(1))
        .otherwise(lit(0))
    )
    .drop("MinDate")
)


In [ ]:
display(df_encounters_transformed)

In [ ]:
#abfss://Healthcare_DE_Project@onelake.dfs.fabric.microsoft.com/LH_Silver.Lakehouse/Files/encounters.csv

(
    df_encounters_transformed.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "Healthcare_DE_Project.HC_Gold.dbo.FACT_Encounters"
    )
)


In [ ]:
l=spark.conf.get('spark.executorEnv.HC_Goldpath')
display(l)

In [ ]:

path = spark.conf.get("spark.executorEnv.HC_Goldpath")

for table_path in [
    f"{path}/Tables/FACT_Encounters",
    f"{path}/Tables/dbo/FACT_Encounters"
]:
    try:
        df_test = spark.read.format("delta").load(table_path)
        print("SUCCESS:", table_path)
        print("Rows:", df_test.count())
    except Exception as e:
        print("FAILED:", table_path)
        print(str(e)[:300])


In [ ]:
print(df_encounters_transformed.count())